# Dendrite spine distance methods comparison

This notebook compares four distance models between spine attachment points: baseline cylindrical distance, skeleton stem approximation, shortest path over original mesh edges, and Heat Method geodesic approximation.

In [1]:
from pathlib import Path

import pandas as pd

from dendrite_analysis import Dendrite, reset_saved_data, set_output_dir, summarize_distance_results
from notebook_widgets import SpineMeshDataset

d:\Kate\anaconda\envs\spinetool\lib\site-packages\libpysal\weights\util.py:23: UserWarning: geopandas not available. Some functionality will be disabled.
  warn("geopandas not available. Some functionality will be disabled.")


In [ ]:
DATASET_PATHS = [
    Path("example_dendrite4"),
]

OUTPUT_DIR = Path("output_dendrite_distance_comparison/4")
METHODS = (
    "cylinder",
    "stem_graph",
    "mesh_graph",
    # "heat",
)
SPINE_FILE_PATTERN = "spine_*.off"

: 

In [ ]:
from dendrite_analysis.mesh_repair import repair_mesh
from spine_analysis.mesh.utils import _mesh_to_v_f, write_off


def save_repaired_mesh(repaired_mesh, output_path):
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    if hasattr(repaired_mesh, "export"):
        repaired_mesh.export(str(output_path))
        return output_path

    vertices, faces = _mesh_to_v_f(repaired_mesh)
    with output_path.open("w") as fd:
        write_off(fd, vertices, faces)
    return output_path


try:
    from tqdm.auto import tqdm
    _tqdm_available = True
except ImportError:
    _tqdm_available = False

all_summaries = []

dataset_iter = tqdm(DATASET_PATHS, desc="Datasets", unit="dataset") if _tqdm_available else DATASET_PATHS
for dataset_path in dataset_iter:
    dataset = SpineMeshDataset().load(str(dataset_path), spine_file_pattern=SPINE_FILE_PATTERN)

    dendrite_items = list(dataset.dendrite_meshes.items())
    dendrite_iter = (
        tqdm(dendrite_items, desc=f"{dataset_path.name} — dendrites", unit="dendrite", leave=False)
        if _tqdm_available
        else dendrite_items
    )

    for dendrite_mesh_name, dendrite_mesh in dendrite_iter:
        repaired_mesh, report_before, report_after = repair_mesh(dendrite_mesh, verbose=True)

        print("\nBefore:", report_before)
        print("\nAfter:", report_after)

        dendrite_name = Path(dendrite_mesh_name).stem
        if not _tqdm_available:
            print(f"[{dataset_path.name}] Processing dendrite: {dendrite_name}")
        dendrite_output_dir = OUTPUT_DIR / dataset_path.name / dendrite_name
        repaired_mesh_path = dendrite_output_dir / f"repaired_{Path(dendrite_mesh_name).name}"
        save_repaired_mesh(repaired_mesh, repaired_mesh_path)
        print(f"Saved repaired mesh to: {repaired_mesh_path}")

        reset_saved_data()
        set_output_dir(str(dendrite_output_dir / "metrics"))

        spine_meshes = {
            spine_name: spine_mesh
            for spine_name, spine_mesh in dataset.spine_meshes.items()
            if dataset.spine_to_dendrite[spine_name] == dendrite_mesh_name
        }

        dendrite = Dendrite(dendrite_name, {dendrite_mesh_name: dendrite_mesh}, spine_meshes)
        results = dendrite.calculate_spine_distance_matrices(
            methods=METHODS,
            output_dir=str(dendrite_output_dir / "distance_methods"),
        )

        summary = summarize_distance_results(results)
        summary.insert(0, "dendrite", dendrite_name)
        summary.insert(0, "dataset", str(dataset_path))
        all_summaries.append(summary)

comparison_summary = pd.concat(all_summaries, ignore_index=True) if all_summaries else pd.DataFrame()
comparison_summary



Datasets:   0%|          | 0/1 [00:00<?, ?dataset/s]

spine_name example_dendrite4\spine_0.off
spine_name example_dendrite4\spine_1.off
spine_name example_dendrite4\spine_10.off
spine_name example_dendrite4\spine_11.off
spine_name example_dendrite4\spine_12.off
spine_name example_dendrite4\spine_13.off
spine_name example_dendrite4\spine_14.off
spine_name example_dendrite4\spine_15.off
spine_name example_dendrite4\spine_16.off
spine_name example_dendrite4\spine_17.off
spine_name example_dendrite4\spine_18.off
spine_name example_dendrite4\spine_19.off
spine_name example_dendrite4\spine_2.off
spine_name example_dendrite4\spine_20.off
spine_name example_dendrite4\spine_21.off
spine_name example_dendrite4\spine_22.off
spine_name example_dendrite4\spine_23.off
spine_name example_dendrite4\spine_24.off
spine_name example_dendrite4\spine_25.off
spine_name example_dendrite4\spine_26.off
spine_name example_dendrite4\spine_27.off
spine_name example_dendrite4\spine_28.off
spine_name example_dendrite4\spine_29.off
spine_name example_dendrite4\spine_3.

example_dendrite4 — dendrites:   0%|          | 0/1 [00:00<?, ?dendrite/s]

=== Before repair ===
Mesh report:
  Vertices              : 57510
  Faces                 : 114879
  Valid (lib check)     : True
  Closed / watertight   : False
  Boundary halfedges    : 187
  Volume negative       : False
  Self-intersections    : True
  Degenerate faces      : 75
  Issues detected       :
    ✗ Mesh has 187 boundary halfedges (93 boundary edges) — open holes exist
    ✗ Mesh is not closed — volume is undefined
    ✗ Mesh has self-intersecting face pairs
    ✗ 75 degenerate (zero-area) faces
  [1/4] CGAL stitch_borders skipped (cannot pickle 'SwigPyObject' object).
  [2/4] Removed 75 degenerate faces.
  [3/4] Hole filling attempted; mesh is still not watertight.
  [4/4] Non-watertight mesh: 49% inward normals (threshold 75%) — no global flip applied.
  [pre-convert] trimesh: not watertight — volume undefined.
  Converting 57581 vertices / 115166 faces to Polyhedron_3 (may take a moment for large meshes)...


In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
comparison_summary.to_csv(OUTPUT_DIR / "all_distance_methods_summary.csv", index=False)
comparison_summary

,dataset,dendrite,method,elapsed_seconds,n_points,mean_distance,median_distance,max_distance
0,example_dendrite,surface_mesh,cylinder,0.007647,22,14.608155,13.225813,37.092314
1,example_dendrite,surface_mesh,mesh_graph,0.990088,22,20.056489,18.301545,49.464770
2,example_dendrite,surface_mesh,stem_graph,6.505217,22,13.861903,12.923086,32.800905
